# 01 — Data Preparation and Cleaning

## Ficha técnica del sistema

| Campo | Detalle |
|---|---|
| **Dominio** | Preguntas y respuestas médicas orientadas a pacientes. |
| **Objetivo** | Construir un asistente RAG que responda preguntas médicas fundamentándose en MedQuAD y conserve trazabilidad de fuente. |
| **Corpus** | `HoangHa/MedQuaD` vía Hugging Face `datasets`. |
| **Embeddings** | `sentence-transformers/all-mpnet-base-v2` (768 dimensiones). |
| **LLM** | `Qwen/Qwen2.5-7B-Instruct`, cuantizado en 4-bit NF4. |
| **Retrieval** | FAISS Flat IP, FAISS HNSW, BM25 híbrido y reranker Cross-Encoder. |
| **Evaluación** | Grounding, Recall@k, Precision@k y RAGAS. |

## Arquitectura dividida en 5 notebooks

```text
01_data_preparation_and_cleaning.ipynb
    └─ medqa_clean.parquet
    └─ medqa_train.parquet
    └─ medqa_val.parquet
    └─ medqa_test.parquet
            │
            ├─> 02_chunking_embeddings_and_indexing.ipynb
            │       └─ df_chunks_A_256.parquet / embeddings_A_256.npy
            │       └─ df_chunks_B_512.parquet / embeddings_B_512.npy
            │       └─ df_chunks_C_1024.parquet / embeddings_C_1024.npy
            │
            ├─> 03_rag_inference_pipeline.ipynb
            ├─> 04_evaluation_and_benchmarks.ipynb
            └─> 05_user_interface_gradio.ipynb
```

> **Orden recomendado:** ejecutar 01 → 02 → 03/04/05. Si cada notebook se abre en una sesión distinta de Google Colab, se deben volver a cargar los archivos `.parquet`/`.npy` requeridos por ese notebook.


# 1.Instalación de librerías

In [ ]:
# Dependencias requeridas por este notebook
!pip install -q pandas pyarrow datasets scikit-learn matplotlib


# 2.Carga y exploración del dataset

## 2.1 Imports y carga

MedQuAD — Dataset

**Origen:** [HoangHa/MedQuaD — Hugging Face](https://huggingface.co/datasets/HoangHa/MedQuaD)  
**Fuente original:** [abachaa/MedQuAD — GitHub](https://github.com/abachaa/MedQuAD)  
**Licencia:** Creative Commons Attribution 4.0 (CC BY)

---

**¿Qué es?**

MedQuAD es un dataset de 47,457 pares pregunta-respuesta construido a partir de
12 sitios web oficiales del NIH, incluyendo cancer.gov, GARD y MedlinePlus.
Cubre 37 tipos de preguntas (tratamiento, diagnóstico, efectos secundarios,
prevalencia, herencia, entre otros) asociadas a enfermedades, medicamentos
y otras entidades médicas.

---

**¿Por qué es relevante para este proyecto?**

Los datasets de QA médico son cruciales para construir sistemas de IA que respondan
preguntas de pacientes sobre síntomas, condiciones y tratamientos, mejorando la
alfabetización en salud y reduciendo la carga de consultas rutinarias sobre los
profesionales médicos.

Para un sistema RAG en particular, MedQuAD ofrece ventajas concretas:

- **Fuentes confiables**: toda la información proviene de instituciones oficiales
  (NIH, CDC, GARD), garantizando calidad y veracidad médica.
- **Preguntas reales de pacientes**: el lenguaje natural de las preguntas refleja
  cómo los usuarios realmente consultan sobre su salud.
- **`question_type` anotado**: permite aplicar stratified splits para evaluación
  balanceada por tipo de pregunta.
- **Licencia abierta**: uso libre con atribución, apto para investigación y publicación.

In [ ]:
import pandas as pd
import numpy as np
import re
from datasets import load_dataset

# Cargamos el dataset (esto es un DatasetDict con uno o más splits)
ds = load_dataset("HoangHa/MedQuaD")

# Vemos qué splits tiene disponibles (train, test, validation, etc.)
print(ds)

# Convertimos el split que nos interese a DataFrame de pandas
df_raw = ds["train"].to_pandas()

print(f"Filas originales : {len(df_raw):,}")
print(f"Columnas         : {df_raw.shape[1]}")
print(f"\nColumnas: {list(df_raw.columns)}")
df_raw.head(2)

## 2.2. Diagnóstico rápido

In [ ]:
print("=== Nulos por columna ===")
null_report = pd.DataFrame({
    "nulos": df_raw.isnull().sum(),
    "% nulos": (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
}).sort_values("% nulos", ascending=False)
print(null_report)

print("\n=== Duplicados ===")
print(f"Filas exactas duplicadas   : {df_raw.duplicated().sum():,}")
print(f"question_id duplicados     : {df_raw['question_id'].duplicated().sum():,}")
print(f"Preguntas duplicadas (texto): {df_raw['question'].duplicated().sum():,}")
print(f"Pares question+answer dup  : {df_raw[['question','answer']].duplicated().sum():,}")

print("\n=== Fuentes sin answer ===")
print(df_raw[df_raw['answer'].isnull()]['document_source'].value_counts())

print("\n=== ESTADÍSTICAS BÁSICAS ===")
display(df_raw.describe(include="all"))

## 2.3. Limpieza

### Paso 1 — Eliminar filas sin `answer` y columnas no críticas
> **Motivo:** 65.4% de las filas no tienen respuesta. Para RAG (tanto indexación de contexto como evaluación de retrieval + generación) necesitamos el par completo pregunta–respuesta.

In [ ]:
df = df_raw.copy()

before = len(df)
df = df[df['answer'].notna()].copy()
# Eliminar columnas no críticas/con muchos nulos
df = df.drop(columns=["synonyms", "umls_cui", "umls_semantic_types", "umls_semantic_group", "category"])
df = df.dropna(subset=["document_id"])
after = len(df)

print(f"Eliminadas {before - after:,} filas sin answer")
print(f"Filas restantes: {after:,}")

### Paso 2 — Eliminar respuestas no informativas
> **Motivo:** Existe al menos una fila con `answer = 'Topics'` (6 chars) que no aporta ningún contenido médico útil.

In [ ]:
import matplotlib.pyplot as plt

df['answer_len'] = df['answer'].str.strip().str.len()

print(df['answer_len'].describe())

df['answer_len'].hist(bins=50, figsize=(12,4))
plt.axvline(100, color='red', linestyle='--', label='100 chars')
plt.axvline(50, color='orange', linestyle='--', label='50 chars')
plt.legend()
plt.title('Distribución de longitud de respuestas')
plt.xlabel('chars')
plt.show()

# Ver ejemplos en el borde del umbral
print(df[df['answer_len'].between(50, 150)][['answer', 'answer_len']].sample(20).to_string())

In [ ]:
garbage_answers = ['frequently asked questions (faqs)', 'topics']
mask_garbage = df['answer'].str.lower().str.strip().isin(garbage_answers)

print(f"A eliminar: {mask_garbage.sum():,}")
display(df[mask_garbage][['question', 'answer', 'document_source']])

before = len(df)
df = df[~mask_garbage].copy()
print(f"Eliminadas: {before - len(df):,} | Restantes: {len(df):,}")

In [ ]:
MIN_ANSWER_LEN = 50  # chars mínimos para considerar una respuesta válida

before = len(df)
short_mask = df['answer'].str.strip().str.len() < MIN_ANSWER_LEN
print(f"Cantidad de respuestas con menos de {MIN_ANSWER_LEN} chars: {short_mask.sum():,}")
print(df[short_mask][['question', 'answer', 'document_source']].to_string())

Nos damos cuenta que varias respuestas tienen como contenido a la pregunta, y esto puede
generar ruido en el modelo de RAG, ya que al recuperar estos chunks el LLM recibirá como
contexto información que no responde nada, llevando a respuestas vacías o incorrectas.

### Paso 3 — Eiminar preguntas en la columna de respuestas

In [ ]:
def contar_respuestas_que_empiezan_con_pregunta(df, col='answer', mostrar_ejemplos=10):
    patron = re.compile(
        r'^\s*(what|how|why|when|where|which|who|whom|whose|is|are|am|was|were|do|does|did|can|could|would|should|will|has|have|had)\b[^?]{0,300}\?',
        flags=re.IGNORECASE
    )

    serie = df[col].fillna('').astype(str)
    mask = serie.apply(lambda x: bool(patron.search(x)))

    total = int(mask.sum())
    print(f"Respuestas que empiezan con pregunta en '{col}': {total}")

    if mostrar_ejemplos > 0 and total > 0:
        print(f"\nEjemplos ({min(mostrar_ejemplos, total)}):")
        for texto in serie[mask].head(mostrar_ejemplos):
            print(f"- {texto[:250]}...\n")

    return mask

In [ ]:
mask_preguntas = contar_respuestas_que_empiezan_con_pregunta(df, col='answer', mostrar_ejemplos=10)

In [ ]:
df[mask_preguntas][['question', 'answer']].head(20)

In [ ]:
def limpiar_answers_con_preguntas_iniciales(df, col_answer='answer', mostrar_ejemplos=10):
    df = df.copy()

    def normalizar_texto(txt):
        txt = '' if txt is None else str(txt)
        txt = txt.strip()
        txt = re.sub(r'\s+', ' ', txt)      # colapsa espacios múltiples
        txt = re.sub(r'\s+\?', '?', txt)    # quita espacios antes de ?
        return txt

    # pregunta inicial válida: solo si aparece al comienzo real del texto
    # o tras signos muy leves de arranque como guiones, viñetas o espacios
    patron_pregunta_inicial = re.compile(
        r'^(?:[-•*:\s"]+)?[^?]{1,500}\?\s*',
        flags=re.IGNORECASE
    )

    def limpiar_inicio_preguntas(txt):
        txt = normalizar_texto(txt)

        while True:
            m = patron_pregunta_inicial.match(txt)
            if not m:
                break

            bloque = m.group(0).strip()

            # Validar que realmente parece pregunta inicial y no texto informativo cualquiera.
            # Exigimos que el bloque empiece con una palabra típica de pregunta en inglés.
            if not re.match(
                r'^(?:[-•*:\s"]+)?'
                r'(what|how|why|when|where|which|who|whom|whose|is|are|am|was|were|do|does|did|can|could|would|should|will|has|have|had)\b',
                bloque,
                flags=re.IGNORECASE
            ):
                break

            txt = txt[m.end():].strip()

        return txt

    a = df[col_answer].fillna('').astype(str)
    a_norm = a.apply(normalizar_texto)
    a_limpio = a_norm.apply(limpiar_inicio_preguntas)

    mask_modificados = a_norm != a_limpio
    print(f"Registros modificados: {int(mask_modificados.sum())}")

    if mostrar_ejemplos > 0 and mask_modificados.sum() > 0:
        print(f"\nEjemplos ({min(mostrar_ejemplos, int(mask_modificados.sum()))}):")
        indices = df.index[mask_modificados][:mostrar_ejemplos]

        for i in indices:
            print(f"- ANTES  : {a_norm.loc[i][:180]}")
            print(f"  DESPUÉS: {a_limpio.loc[i][:180]}\n")

    df[col_answer] = a_limpio

    mask_vacios = df[col_answer].fillna('').astype(str).str.strip().eq('')
    print(f"Registros eliminados por quedar vacíos: {int(mask_vacios.sum())}")

    df = df.loc[~mask_vacios].copy()

    return df

In [ ]:
df = limpiar_answers_con_preguntas_iniciales(df, col_answer='answer')

In [ ]:
before = len(df)
mask = df['answer'].str.startswith('More detailed information', na=False)
print(f"Filas a eliminar: {mask.sum():,}")
print(df[mask][['question', 'answer', 'document_source']].to_string())

df = df[~mask].copy()
print(f"Eliminadas: {before - len(df):,} | Restantes: {len(df):,}")

### Paso 4 — Eliminar pares `question + answer` duplicados
> **Motivo:** 585 pares idénticos introducen bias en evaluación. Conservamos la primera ocurrencia.

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=['question', 'answer'], keep='first').copy()
print(f"Eliminadas {before - len(df):,} filas duplicadas (question+answer)")
print(f"Filas restantes: {len(df):,}")

### Paso 5 — Eliminar preguntas duplicadas (texto)
> **Motivo:** 2,838 preguntas con el mismo texto pero distintas respuestas o fuentes. Conservamos la versión con respuesta más larga (más informativa).

In [ ]:
before = len(df)

# Ordenar por longitud de respuesta descendente antes de deduplicar
df = df.sort_values('answer', key=lambda s: s.str.len(), ascending=False)

# Tabla de preguntas duplicadas con conteo
duplicated_mask = df.duplicated(subset=['question'], keep=False)
print(f"Filas duplicadas totales: {duplicated_mask.sum():,}")

conteo = df[duplicated_mask].groupby('question').size().reset_index(name='repeticiones').sort_values('repeticiones', ascending=False)
display(conteo)

# Muestra de las primeras 50 filas duplicadas
print("\nMuestra de las primeras 50 filas duplicadas:")
display(df[duplicated_mask][['question', 'answer', 'document_source']].sort_values('question').head(50))

In [ ]:
df = df.drop_duplicates(subset=['question'], keep='first').copy()
df = df.sort_index()

print(f"\nEliminadas {before - len(df):,} preguntas duplicadas (texto)")
print(f"Filas restantes: {len(df):,}")
print(f"Preguntas duplicadas (texto): {df['question'].duplicated().sum():,}")

### Paso 6 — Reasignar `question_id` único
> **Motivo:** El campo `question_id` original se reutiliza entre fuentes distintas (ej. `0000001-1` aparece en GHR, NINDS, ADAM, etc.). Para RAG necesitamos IDs globalmente únicos.

In [ ]:
# Guardar el original por trazabilidad
df['question_id_original'] = df['question_id']

# Nuevo ID: source + id_original + índice para garantizar unicidad
df = df.reset_index(drop=True)
df['question_id'] = (
    df['document_source'].str.upper().str.replace(' ', '_') + '_' +
    df['question_id_original'].str.replace('/', '-') + '_' +
    df.index.astype(str)
)

print(f"IDs duplicados después de reasignación: {df['question_id'].duplicated().sum()}")
print("Ejemplo IDs nuevos:")
print(df['question_id'].head(5).values)

### Paso 7 — Normalizar texto de `answer` y `question`
El ruido de whitespace afecta negativamente a los embeddings.

In [ ]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return text
    # Reemplazar tabs por espacio
    text = text.replace('\t', ' ')
    # Colapsar múltiples espacios en uno
    text = re.sub(r' {2,}', ' ', text)
    # Limpiar espacios antes de saltos de línea
    text = re.sub(r' +\n', '\n', text)
    # Colapsar más de 2 saltos de línea consecutivos
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

df['answer']   = df['answer'].apply(normalize_text)
df['question'] = df['question'].apply(normalize_text)

# Verificación
tabs_after = df['answer'].str.contains(r'\t').sum()
double_space_after = df['answer'].str.contains(r'  ').sum()
print(f"Tabs en answer después de limpieza   : {tabs_after}")
print(f"Doble espacio en answer tras limpieza: {double_space_after}")

### Paso 8 — Marcar respuestas genéricas repetidas
> **Motivo:** La respuesta de herencia autosómica recesiva se repite 348 veces. Es válida como contenido médico, pero al usarla como ground truth en evaluación puede sesgar las métricas. La marcamos para poder filtrarla opcionalmente en los sets de evaluación.

In [ ]:
# =========================
# 1. Normalizar respuestas
# =========================
df['answer_norm'] = (
    df['answer']
    .fillna('')
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.lower()
)

# ==========================================================
# 2. Marcar como genéricas las respuestas que se repiten mucho
# ==========================================================
GENERIC_THRESHOLD = 5
answer_counts = df['answer_norm'].value_counts()
generic_answers = set(answer_counts[answer_counts > GENERIC_THRESHOLD].index)

df['answer_is_generic'] = df['answer_norm'].isin(generic_answers)

print(f"Respuestas distintas marcadas como genéricas (>{GENERIC_THRESHOLD} apariciones): {len(generic_answers)}")
print(f"Filas afectadas antes del recorte: {df['answer_is_generic'].sum():,}")
print("\nTop respuestas genéricas (primeros 120 caracteres):")
for ans in list(generic_answers)[:10]:
    print(f"  [{answer_counts[ans]}x] {ans[:120]}...")

# ============================================
# 3. Recortar respuestas genéricas repetidas
# ============================================
df_generic = df[df['answer_is_generic']].copy()
df_non_generic = df[~df['answer_is_generic']].copy()

df_generic_limited = (
    df_generic
    .groupby('answer_norm', group_keys=False)
    .head(GENERIC_THRESHOLD)
    .copy()
)

df = pd.concat([df_non_generic, df_generic_limited], ignore_index=True)
df = df.sort_index()

print("\nTop respuestas repetidas después del recorte:")
top_counts = df['answer_norm'].value_counts().head(10)
for ans, count in top_counts.items():
    print(f"  [{count}x] {ans[:120]}...")

df.drop(columns=['answer_norm'], inplace=True)

# ============================================
# 4. Resumen después del recorte
# ============================================
print(f"\n{'='*50}")
print(f"Filas genéricas antes del recorte : {len(df_generic):,}")
print(f"Filas genéricas después del recorte: {len(df_generic_limited):,}")
print(f"Eliminadas por recorte             : {len(df_generic) - len(df_generic_limited):,}")
print(f"{'='*50}")
print(f"Filas totales finales              : {len(df):,}")

Durante el análisis se detectó que ciertas respuestas médicas válidas aparecen
cientos de veces en el dataset. Por ejemplo, la respuesta sobre herencia
autosómica recesiva se repetía 348 veces con texto idéntico.

Esto es problemático para RAG por dos razones:

1. **Sesgo en evaluación**: si una respuesta domina el dataset, el modelo puede
   obtener buenas métricas simplemente por frecuencia, no por calidad de recuperación.

2. **Redundancia en el índice vectorial**: tener 348 chunks idénticos no aporta
   diversidad semántica y ocupa espacio innecesario.

La solución no fue eliminarlas completamente, ya que son respuestas médicamente
válidas, sino **limitar cada respuesta única a un máximo de 5 apariciones**.
Esto preserva el patrón para que el modelo lo aprenda, sin que distorsione
las métricas de evaluación.

### Paso 9 — Marcar respuestas muy largas
> **Motivo:** El modelo `all-mpnet-base-v2` tiene un límite de 384 tokens
(~1,500 chars aproximados en inglés). Textos que superen este umbral serán
truncados silenciosamente, perdiendo información clave.

El análisis muestra que aproximadamente el 25% del dataset supera los 1,500 chars,
con un máximo de 26,976 chars. Estas respuestas serán marcadas con
`answer_needs_chunking = True`.

In [ ]:
MAX_ANSWER_LEN = 1500

df['answer_len'] = df['answer'].str.len()
df['answer_needs_chunking'] = df['answer_len'] > MAX_ANSWER_LEN

print(f"Respuestas que necesitan chunking (>{MAX_ANSWER_LEN} chars): {df['answer_needs_chunking'].sum():,}")
print(f"\nDistribución de longitud de respuestas:")
print(df['answer_len'].describe().round(0))

print(f"\nDistribución por rangos:")
bins = [0, 500, 1000, 2000, 5000, 10000, float('inf')]
labels = ['0-500', '500-1k', '1k-2k', '2k-5k', '5k-10k', '>10k']
print(df['answer_len'].pipe(pd.cut, bins=bins, labels=labels).value_counts().sort_index())

### Paso 10 — Rellenar `question_focus` nulo
> **Motivo:** 14 filas sin `question_focus`. Se puede inferir del `document_url` o dejar como 'unknown'.

In [ ]:
def infer_focus_from_url(row):
    """Intenta extraer el tema del último segmento de la URL."""
    if pd.notna(row['question_focus']):
        return row['question_focus']
    try:
        slug = row['document_url'].rstrip('/').split('/')[-1]
        # quitar extensiones y reemplazar guiones
        slug = re.sub(r'\.(html|htm|aspx)$', '', slug)
        return slug.replace('-', ' ').replace('_', ' ').strip() or 'unknown'
    except Exception:
        return 'unknown'

null_focus_before = df['question_focus'].isnull().sum()
df['question_focus'] = df.apply(infer_focus_from_url, axis=1)
null_focus_after = df['question_focus'].isnull().sum()

print(f"question_focus nulos antes : {null_focus_before}")
print(f"question_focus nulos después: {null_focus_after}")

In [ ]:
df.describe(include="all")

### Paso 11 — Construir splits train / val / test
> Para RAG se recomienda separar los splits **antes** de indexar. Las preguntas de val/test no deben estar en el índice de documentos.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Excluir respuestas genéricas del set de evaluación
df_eval_pool = df[~df['answer_is_generic']].copy()
df_generic   = df[df['answer_is_generic']].copy()

use_stratify = df_eval_pool['question_type'].value_counts().min() >= 2

train_val, test = train_test_split(
    df_eval_pool,
    test_size=0.1,
    random_state=42,
    stratify=df_eval_pool['question_type'] if use_stratify else None
)

use_stratify_2 = train_val['question_type'].value_counts().min() >= 2

train, val = train_test_split(
    train_val,
    test_size=0.111,
    random_state=42,
    stratify=train_val['question_type'] if use_stratify_2 else None
)

train = pd.concat([train, df_generic], ignore_index=True)

print(f"Train : {len(train):,} ({len(train)/len(df)*100:.1f}%)")
print(f"Val   : {len(val):,} ({len(val)/len(df)*100:.1f}%)")
print(f"Test  : {len(test):,} ({len(test)/len(df)*100:.1f}%)")

### Resumen final y guardado

In [ ]:
print("=" * 50)
print("RESUMEN DEL PROCESO DE LIMPIEZA")
print("=" * 50)
print(f"Filas originales              : {len(df_raw):>8,}")
print(f"  - Sin answer (eliminadas)   : {len(df_raw[df_raw['answer'].isna()]):>8,}")
print(f"  - Pares duplicados (elim.)  : {585:>8,}")
print(f"  - Preguntas duplicadas      : {2838:>8,}")
print(f"Filas limpias totales         : {len(df):>8,}")
print(f"  - Marcadas como genéricas   : {df['answer_is_generic'].sum():>8,}")
print(f"  - Necesitan chunking        : {df['answer_needs_chunking'].sum():>8,}")
print()
print("Columnas nuevas añadidas:")
new_cols = ['question_id_original', 'answer_is_generic', 'answer_needs_chunking', 'answer_len']
for c in new_cols:
    print(f"  + {c}")

print("\nDistribución final por fuente:")
print(df['document_source'].value_counts())

In [ ]:
# Guardar archivos limpios
df.to_parquet("medqa_clean.parquet", index=False)
train.to_parquet("medqa_train.parquet", index=False)
val.to_parquet("medqa_val.parquet",   index=False)
test.to_parquet("medqa_test.parquet", index=False)

print("Archivos guardados:")
print("  medqa_clean.parquet  — dataset limpio completo")
print("  medqa_train.parquet  — split de entrenamiento/indexación")
print("  medqa_val.parquet    — split de validación")
print("  medqa_test.parquet   — split de test (evaluación final RAG)")